# 7.8 EmbeddingGemma fine-tuning v2 (no test leakage)
Replaces 7.3–7.7 for the fine-tuned model. Triplets come ONLY from the classification `train` split; no evaluation on test during training; final model after 1 epoch is used.

Run on Colab with a GPU. Secrets needed: `HF_TOKEN`. Run all cells top to bottom and keep outputs.

In [ ]:
!pip install -q -U sentence-transformers git+https://github.com/huggingface/transformers@v4.56.0-Embedding-Gemma-preview datasets scikit-learn

In [ ]:
from google.colab import userdata
from huggingface_hub import login
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

## 1. Load the proper splits (same as all other models)

In [ ]:
import random, numpy as np, pandas as pd, torch
from datasets import load_dataset, Dataset, DatasetDict

ds = load_dataset("dinushiTJ/nz_research_commons_classification")
lab = ds["train"].features["classification_label"].int2str

def to_df(split):
    d = ds[split].to_pandas()
    d["label_text"] = d["classification_label"].apply(lab)
    return d[["embedding_text", "label_text", "record_id_hash"]]

train_df = to_df("train")
test_df = to_df("test")
print(len(train_df), len(test_df))                 # expect 13271 3318
print(train_df["label_text"].value_counts())       # expect 1469 / 11802
assert not set(train_df.record_id_hash) & set(test_df.record_id_hash), "train/test overlap!"
print("OK: no record overlap between train and test")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/83.2M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/13271 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3318 [00:00<?, ? examples/s]

13271 3318
label_text
non_maori_origin    11802
maori_origin         1469
Name: count, dtype: int64
OK: no record overlap between train and test


## 2. Build triplets (same function as 7.3, now seeded)

In [ ]:
def build_triplets(df, num_triplets_per_sample=2):
    triplets = []
    label_to_texts = {label: g["embedding_text"].tolist() for label, g in df.groupby("label_text")}
    all_labels = list(label_to_texts.keys())
    for label in all_labels:
        positives_pool = label_to_texts[label]
        negatives_pool = [t for l in all_labels if l != label for t in label_to_texts[l]]
        if len(positives_pool) < 2:
            continue
        for anchor in positives_pool:
            for _ in range(num_triplets_per_sample):
                positive = anchor
                while positive == anchor:
                    positive = random.choice(positives_pool)
                negative = random.choice(negatives_pool)
                triplets.append({"anchor": anchor, "positive": positive, "negative": negative, "label": label})
    return pd.DataFrame(triplets)

random.seed(42)
train_triplets = build_triplets(train_df, num_triplets_per_sample=2)
test_triplets = build_triplets(test_df, num_triplets_per_sample=1)
print(len(train_triplets), len(test_triplets))     # expect 26542 3318

## 3. 10k pool -> 5k subset (same rule as 7.4)

In [ ]:
maori_all = train_triplets[train_triplets["label"] == "maori_origin"]
non_maori_all = train_triplets[train_triplets["label"] != "maori_origin"]
pool = pd.concat([maori_all, non_maori_all.sample(n=10000 - len(maori_all), random_state=42)])
print("Pool:", len(pool), "| Maori:", (pool.label == "maori_origin").sum(), "| Non-Maori:", (pool.label != "maori_origin").sum())  # expect 10000 / 2938 / 7062

maori_df = pool[pool["label"] == "maori_origin"]
non_maori_df = pool[pool["label"] != "maori_origin"]
new_train_df = pd.concat([maori_df, non_maori_df.sample(n=5000 - len(maori_df), random_state=42)])
new_train_df = new_train_df.sample(frac=1, random_state=42).reset_index(drop=True)
print("Final train size:", len(new_train_df), "| Maori:", (new_train_df.label == "maori_origin").sum())  # expect 5000 / 2938

In [ ]:
# Leakage check: no test record may appear anywhere in the training triplets
test_texts = set(test_df["embedding_text"])
used = set(new_train_df["anchor"]) | set(new_train_df["positive"]) | set(new_train_df["negative"])
leak = len(test_texts & used)
print("Test records appearing in training triplets:", leak)   # MUST be 0
assert leak == 0

In [ ]:
DatasetDict({
    "train": Dataset.from_pandas(train_triplets, preserve_index=False),
    "test": Dataset.from_pandas(test_triplets, preserve_index=False),
}).push_to_hub("dinushiTJ/nz_research_commons_embedding_triplets_v2")
DatasetDict({
    "train": Dataset.from_pandas(new_train_df, preserve_index=False),
}).push_to_hub("dinushiTJ/nz_research_commons_embedding_triplets_5k_v2")

## 4. Fine-tune (same settings as 7.5, no test evaluation, no checkpoint selection)

In [ ]:
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.sentence_transformer.losses import TripletLoss, TripletDistanceMetric

task_name = "Classification"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("google/embeddinggemma-300M").to(device=DEVICE)

trainset = Dataset.from_pandas(new_train_df, preserve_index=False).map(
    lambda e: {"anchor": e["label"], "positive": e["positive"], "negative": e["negative"]},
    remove_columns=["label"])
print(trainset)                                   # expect 5000 rows
print(trainset[0]["anchor"])                      # label text, as in 7.5

modules.json:   0%|          | 0.00/573 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/997 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

2_Dense/model.safetensors:   0%|          | 0.00/9.44M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

3_Dense/model.safetensors:   0%|          | 0.00/9.44M [00:00<?, ?B/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Dataset({
    features: ['anchor', 'positive', 'negative'],
    num_rows: 5000
})
maori_origin


In [ ]:
args = SentenceTransformerTrainingArguments(
    output_dir="rc-embedding-gemma-v2",
    prompts=model.prompts[task_name],
    num_train_epochs=1,
    per_device_train_batch_size=1,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    logging_steps=50,
    eval_strategy="no",
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    seed=42,
    report_to="none",
)
loss = TripletLoss(model, distance_metric=TripletDistanceMetric.COSINE, triplet_margin=0.3)
trainer = SentenceTransformerTrainer(model=model, args=args, train_dataset=trainset, loss=loss)
out = trainer.train()
print(out)                                         # record global_step and train_loss

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
50,0.151100
100,0.042600
150,0.014700
200,0.020300
250,0.018000
300,0.020500
350,0.071800
400,0.043200


Step,Training Loss
50,0.151100
100,0.042600
150,0.014700
200,0.020300
250,0.018000
300,0.020500
350,0.071800
400,0.043200
450,0.046800
500,0.020800


In [ ]:
trainer.model.save_pretrained("rc-embedding-gemma-v2-final")
trainer.model.push_to_hub("dinushiTJ/nz-research-commons-embedding-gemma-v2")
print("Pushed final model after 1 epoch")

## 5. Inference on the 3,318 test records (same method as 7.6)

In [ ]:
ds_test = load_dataset("dinushiTJ/nz_research_commons_inference_results", split="train")
label_feature = ds_test.features["classification_label"]
class_names = label_feature.names
TASK_NAME = "Classification"
CONTEXT_LEN = 2048

ft_model = SentenceTransformer("dinushiTJ/nz-research-commons-embedding-gemma-v2").to(device=DEVICE)
ft_model.eval()
ft_label_embeddings = ft_model.encode(class_names, prompt_name=TASK_NAME, convert_to_numpy=True, normalize_embeddings=True)

def embed_document(text, token_count, model):
    if token_count <= CONTEXT_LEN:
        return model.encode(text, prompt_name=TASK_NAME, convert_to_numpy=True, normalize_embeddings=True)
    words = text.split()
    chunk_size = max(1, int(len(words) * CONTEXT_LEN / token_count))
    embs = [model.encode(" ".join(words[i:i + chunk_size]), prompt_name=TASK_NAME, convert_to_numpy=True, normalize_embeddings=True)
            for i in range(0, len(words), chunk_size)]
    return np.mean(np.stack(embs, axis=0), axis=0)

def classify(example):
    sims = np.dot(embed_document(example["embedding_text"], example["embedding_token_count"], ft_model), ft_label_embeddings.T)
    i = int(np.argmax(sims))
    return {"FTv2EmbGemZeroLabel": class_names[i], "FTv2EmbGemZeroID": i, "FTv2EmbGemZeroScore": float(sims[i])}

ds_test = ds_test.map(classify, batched=False)
ds_test.push_to_hub("dinushiTJ/nz_research_commons_inference_results", split="train")

## 6. Evaluation (same metrics as 7.7)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score
y_true = ds_test["classification_label"]
y_pred = [label_feature.str2int(l) for l in ds_test["FTv2EmbGemZeroLabel"]]
print("Confusion matrix (rows=reference, cols=pred; order:", class_names, ")")
print(confusion_matrix(y_true, y_pred, labels=[0, 1]))
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(f"F1 Macro: {f1_score(y_true, y_pred, average='macro'):.4f}")
print("Disagreements:", sum(t != p for t, p in zip(y_true, y_pred)))